# Entrega 04 — dados, qualidade e baselines do RU

**Equipe 02:** Letícia Braz, Bianca Lavine, Kaio Vitor e Maria Dionila.  
**Repositório:** https://github.com/LeticiaBrazBonfim/previsao-demanda-ru-ufpb

Este é o **único notebook coletivo** da Entrega 4. Os notebooks dos integrantes registram suas contribuições separadamente.

## Contrato antes de mexer nos dados

| Pergunta | Resposta neste piloto |
|---|---|
| O que prevemos? | Quantidade de refeições **servidas** em uma data, tipo de refeição e curso. |
| O que é uma linha? | Uma combinação `data + tipo_refeicao + curso`. Essa é a chave da base final. |
| Quando prevemos? | Na véspera (`D-1`), para o dia seguinte (`D`). |
| Qual é a fonte do alvo? | IDs distintos de eventos de catraca com `situacao_evento = servida`. |
| Quem está coberto? | Seis cursos técnicos fictícios e uma unidade fictícia, de 01/01/2019 a 14/11/2026. |
| De onde vêm os dados? | Três CSVs sintéticos: catraca, estudantes e componentes. |
| Quando são atualizados? | No mock, a extração é completa para todo o período. No RU real, frequência e atraso ainda precisam ser confirmados. |
| Há restrições? | Dados reais exigem autorização, proteção dos identificadores e revisão de finalidade conforme a LGPD. |

**Importante:** o mock não contém preço nem custo. Se o arquivo real tiver um valor por refeição servida, ele poderá ser somado por curso após a equipe confirmar o significado da coluna. Aqui não inventamos R$ nem calculamos economia ou subsídio. Os números abaixo **não** descrevem o RU da UFPB.

## 1. Ler as três fontes

Altere as configurações abaixo quando houver dados reais. Os arquivos reais também precisarão ter suas **colunas e significados conferidos**; trocar só o caminho não garante compatibilidade. Guarde sempre uma cópia intocada dos originais.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Procuramos a raiz para o notebook funcionar tanto nela quanto em notebooks/.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra o notebook a partir da pasta do projeto.")

MODO = "mock"  # Para dados reais, use "real" e leia as instruções da seção 3.
BRUTOS = RAIZ / "dados/bruto/mock"
COBERTURA_REAL = RAIZ / "dados/bruto/cobertura_real.csv"
COLUNA_VALOR_EVENTO = None  # Exemplo futuro: "valor_pago", após confirmar o significado.
PROCESSADOS = RAIZ / "dados/processado"
FIM_TREINO = pd.Timestamp("2024-12-31")
FIM_VALIDACAO = pd.Timestamp("2025-12-31")

catraca = pd.read_csv(BRUTOS / "catraca_ru.csv", dtype="string")
estudantes = pd.read_csv(BRUTOS / "estudantes.csv", dtype="string")
componentes = pd.read_csv(BRUTOS / "matriculas_componentes.csv", dtype="string")
print("Linhas brutas:", {"catraca": len(catraca), "estudantes": len(estudantes), "componentes": len(componentes)})


Linhas brutas: {'catraca': 224880, 'estudantes': 2016, 'componentes': 5922}


## 2. Conferir o que chegou

Checamos colunas, chaves, datas e vínculos antes de calcular o alvo. Valores faltantes em campos **não essenciais** são mostrados para análise, sem eliminar linhas automaticamente. As datas e categorias fixas do gerador são conferidas só no modo `mock`.

In [2]:
obrigatorias = {
    "catraca": {"id_registro_utilizacao", "id_matricula", "data_hora_refeicao", "tipo_refeicao", "curso", "situacao_evento"},
    "estudantes": {"id_matricula", "periodo_letivo", "curso"},
    "componentes": {"id_matricula_componente", "id_matricula", "periodo_letivo"},
}
for nome, tabela in [("catraca", catraca), ("estudantes", estudantes), ("componentes", componentes)]:
    faltam = obrigatorias[nome] - set(tabela.columns)
    if faltam:
        raise ValueError(f"Faltam colunas em {nome}: {sorted(faltam)}. Adapte a leitura ao esquema real.")

catraca["data"] = pd.to_datetime(catraca["data_hora_refeicao"], errors="coerce").dt.normalize()

def linhas_vazias(tabela, colunas):
    campos = tabela[colunas]
    return int((campos.isna() | campos.apply(lambda c: c.str.strip().eq(""))).any(axis=1).sum())

problemas = {
    "ID de evento vazio": linhas_vazias(catraca, ["id_registro_utilizacao"]),
    "ID de evento repetido": catraca.duplicated("id_registro_utilizacao").sum(),
    "data inválida": catraca.data.isna().sum(),
    "refeição, curso ou vínculo vazio": linhas_vazias(catraca, ["tipo_refeicao", "curso", "id_matricula"]),
    "situação não reconhecida": (~catraca.situacao_evento.isin(["servida", "negada"])).sum(),
    "estudante/período/curso vazio": linhas_vazias(estudantes, ["id_matricula", "periodo_letivo", "curso"]),
    "estudante/período repetido": estudantes.duplicated(["id_matricula", "periodo_letivo"]).sum(),
    "chave de componente vazia": linhas_vazias(componentes, ["id_matricula_componente", "id_matricula", "periodo_letivo"]),
    "ID de componente repetido": componentes.duplicated("id_matricula_componente").sum(),
    "evento sem vínculo cadastrado": (~catraca.id_matricula.isin(estudantes.id_matricula)).sum(),
}
chaves_componentes = componentes[["id_matricula", "periodo_letivo"]].drop_duplicates()
chaves_estudantes = estudantes[["id_matricula", "periodo_letivo"]].drop_duplicates()
ligacao = chaves_componentes.merge(chaves_estudantes, how="left", indicator=True,
                                   on=["id_matricula", "periodo_letivo"])
problemas["componente sem estudante/período"] = ligacao._merge.ne("both").sum()

# Antes de prever por curso, conferimos se o curso da catraca bate com o cadastro.
# Um vínculo que muda de curso exige uma associação temporal, não uma escolha arbitrária.
if estudantes.groupby("id_matricula").curso.nunique(dropna=False).gt(1).any():
    raise ValueError("Há vínculo com mais de um curso: associe pelo período antes de usar dados reais.")
cadastro_cursos = estudantes[["id_matricula", "curso"]].drop_duplicates("id_matricula")
confronto = catraca[["id_matricula", "curso"]].merge(
    cadastro_cursos, how="left", on="id_matricula", validate="many_to_one",
    suffixes=("_evento", "_cadastro"))
problemas["linhas alteradas na conferência de curso"] = abs(len(confronto) - len(catraca))
problemas["curso da catraca diferente do cadastro"] = confronto.curso_evento.ne(confronto.curso_cadastro).sum()
display(pd.DataFrame({"problema": problemas.keys(), "linhas": problemas.values()}))
if any(valor > 0 for valor in problemas.values()):
    raise ValueError("Há problemas nas chaves ou datas. Corrija a origem ou documente o tratamento antes de continuar.")

if MODO == "mock":
    assert catraca.data.min() == pd.Timestamp("2019-01-01")
    assert catraca.data.max() == pd.Timestamp("2026-11-14")
    assert set(catraca.tipo_refeicao) == {"cafe_da_manha", "almoco", "jantar"}

# Resumo com denominador: ajuda a localizar ausências antes de decidir o que fazer.
perfil = []
for nome, tabela in [("catraca", catraca), ("estudantes", estudantes), ("componentes", componentes)]:
    for coluna in tabela.columns:
        n = int(tabela[coluna].isna().sum())
        if n:
            perfil.append({"arquivo": nome, "coluna": coluna, "faltantes": n,
                           "linhas": len(tabela), "taxa": round(n / len(tabela), 4)})
print("Campos com valores ausentes:")
display(pd.DataFrame(perfil) if perfil else "Nenhum no mock atual")
print("Tipos após leitura e conversão da data:")
display(pd.DataFrame({nome: tabela.dtypes.astype(str).value_counts()
                      for nome, tabela in [("catraca", catraca), ("estudantes", estudantes), ("componentes", componentes)]}).fillna(0).astype(int))
print("Eventos por situação, refeição e curso:")
for coluna in ["situacao_evento", "tipo_refeicao", "curso"]:
    display(catraca[coluna].value_counts().rename(coluna))

,problema,linhas
0,ID de evento vazio,0
1,ID de evento repetido,0
2,data inválida,0
3,"refeição, curso ou vínculo vazio",0
4,situação não reconhecida,0
5,estudante/período/curso vazio,0
6,estudante/período repetido,0
7,chave de componente vazia,0
8,ID de componente repetido,0
9,evento sem vínculo cadastrado,0


Campos com valores ausentes:


'Nenhum no mock atual'

Tipos após leitura e conversão da data:


,catraca,estudantes,componentes
datetime64[us],1,0,0
string,13,18,18


Eventos por situação, refeição e curso:


situacao_evento
servida    221600
negada       3280
Name: situacao_evento, dtype: Int64

tipo_refeicao
almoco           111807
cafe_da_manha     59601
jantar            53472
Name: tipo_refeicao, dtype: Int64

curso
CURSO_TECNICO_B    46134
CURSO_TECNICO_D    46122
CURSO_TECNICO_F    46108
CURSO_TECNICO_A    29034
CURSO_TECNICO_C    28870
CURSO_TECNICO_E    28612
Name: curso, dtype: Int64

## 3. Distinguir zero de falta de dados

**Zero** quer dizer que houve cobertura confiável e nenhuma refeição servida naquela combinação. **Sem dado** quer dizer que a extração ou o calendário não permitem saber a quantidade.

- **Mock:** o gerador cobre todo o intervalo, as três refeições e os seis cursos. Ele simula fechamento aos domingos. Por isso, podemos completar sua grade e marcar ausência de evento como zero.
- **Dados reais:** prepare `cobertura_real.csv` com uma linha por `data, tipo_refeicao, curso`, além de `aberto` e `extracao_completa` (`sim`/`nao`). Inclua apenas cursos ativos naquela data. Se a extração não estiver completa, a quantidade fica vazia e não entra na avaliação. **Sem esse arquivo, o notebook para**, evitando inventar zeros.

O calendário e a integridade da extração precisam ser confirmados com o RU. O valor, se existir, deve ser o valor **de uma refeição servida por evento**; informe o nome da coluna em `COLUNA_VALOR_EVENTO` apenas após conferir isso. Valor cobrado, custo contratual e subsídio não são a mesma coisa.

In [3]:
chave = ["data", "tipo_refeicao", "curso"]
servidas = catraca.loc[catraca.situacao_evento.eq("servida")].copy()
contagens = (servidas.groupby(chave).id_registro_utilizacao.nunique()
             .reset_index(name="quantidade_observada"))

if MODO == "mock":
    datas = pd.date_range("2019-01-01", "2026-11-14", freq="D")
    refeicoes = sorted(catraca.tipo_refeicao.unique())
    cursos = sorted(estudantes.curso.dropna().unique())
    cobertura = pd.MultiIndex.from_product([datas, refeicoes, cursos], names=chave).to_frame(index=False)
    cobertura["aberto"] = cobertura.data.dt.day_of_week.lt(6)
    cobertura["extracao_completa"] = True
elif MODO == "real":
    cobertura = pd.read_csv(COBERTURA_REAL, dtype="string")
    faltam = set(chave + ["aberto", "extracao_completa"]) - set(cobertura.columns)
    if faltam:
        raise ValueError(f"Faltam colunas no calendário de cobertura: {sorted(faltam)}")
    cobertura["data"] = pd.to_datetime(cobertura.data, errors="raise")
    for coluna in ["aberto", "extracao_completa"]:
        if not cobertura[coluna].isin(["sim", "nao"]).all():
            raise ValueError(f"Use somente sim/nao na coluna {coluna}.")
        cobertura[coluna] = cobertura[coluna].eq("sim")
else:
    raise ValueError("MODO deve ser mock ou real.")

if cobertura.duplicated(chave).any() or cobertura[chave].isna().any().any():
    raise ValueError("A cobertura precisa ter chave única e sem vazios.")
fora = contagens.merge(cobertura[chave], how="left", indicator=True, on=chave)
if fora._merge.ne("both").any():
    raise ValueError("Há refeições servidas sem linha correspondente na cobertura. Confira cursos e calendário.")

base_curso = cobertura.merge(contagens, how="left", validate="one_to_one", on=chave)
if base_curso.loc[~base_curso.aberto, "quantidade_observada"].notna().any():
    raise ValueError("Há refeições em dia marcado como fechado. Revise o calendário.")

# Só uma extração completa autoriza preencher ausência de evento com zero.
base_curso["quantidade_servida"] = base_curso.quantidade_observada.where(base_curso.extracao_completa)
base_curso.loc[base_curso.extracao_completa, "quantidade_servida"] = (
    base_curso.loc[base_curso.extracao_completa, "quantidade_servida"].fillna(0))
base_curso["quantidade_servida"] = base_curso.quantidade_servida.astype("Int64")
base_curso = base_curso.drop(columns="quantidade_observada")

# Campo monetário opcional. O mock não o possui e não recebe preço inventado.
if COLUNA_VALOR_EVENTO is not None:
    if COLUNA_VALOR_EVENTO not in servidas:
        raise ValueError("A coluna de valor informada não existe na catraca.")
    servidas["valor_evento"] = pd.to_numeric(servidas[COLUNA_VALOR_EVENTO], errors="coerce")
    if servidas.valor_evento.isna().any() or servidas.valor_evento.lt(0).any():
        raise ValueError("Há valor de refeição servida vazio, inválido ou negativo.")
    valores = servidas.groupby(chave).valor_evento.sum().reset_index(name="valor_observado")
    base_curso = base_curso.merge(valores, how="left", validate="one_to_one", on=chave)
    base_curso.loc[~base_curso.extracao_completa, "valor_observado"] = np.nan
    base_curso.loc[base_curso.extracao_completa, "valor_observado"] = (
        base_curso.loc[base_curso.extracao_completa, "valor_observado"].fillna(0))
else:
    base_curso["valor_observado"] = np.nan

# O total do RU é derivado dos cursos; se algum curso estiver incompleto, o total fica vazio.
base_diaria = (base_curso.groupby(["data", "tipo_refeicao"])
               .agg(quantidade_servida=("quantidade_servida", "sum"),
                    cobertura_completa=("extracao_completa", "all"),
                    aberto=("aberto", "all"))
               .reset_index())
base_diaria.loc[~base_diaria.cobertura_completa, "quantidade_servida"] = pd.NA
base_diaria["quantidade_servida"] = base_diaria.quantidade_servida.astype("Int64")

if MODO == "mock":
    assert base_curso.quantidade_servida.sum() == len(servidas)
    assert base_diaria.quantidade_servida.sum() == len(servidas)
    assert len(base_curso) == len(datas) * len(refeicoes) * len(cursos)

PROCESSADOS.mkdir(parents=True, exist_ok=True)
base_curso.to_csv(PROCESSADOS / "base_analitica_diaria_curso.csv", index=False)
base_diaria.to_csv(PROCESSADOS / "base_analitica_diaria.csv", index=False)
print("Refeições servidas:", len(servidas), "| linhas por curso:", len(base_curso))
print("Zeros conhecidos:", int(base_curso.quantidade_servida.eq(0).sum()),
      "| linhas sem cobertura:", int(base_curso.quantidade_servida.isna().sum()))
print("Distribuição da quantidade em dias abertos e completos:")
display(base_curso.loc[base_curso.aberto & base_curso.extracao_completa]
        .groupby("tipo_refeicao").quantidade_servida.agg(["count", "mean", "median", "max"]).round(2))
display(base_curso.head())

Refeições servidas: 221600 | linhas por curso: 51750
Zeros conhecidos: 11990 | linhas sem cobertura: 0
Distribuição da quantidade em dias abertos e completos:


,count,mean,median,max
tipo_refeicao,,,,
almoco,14790,7.45,8.0,19
cafe_da_manha,14790,3.97,4.0,14
jantar,14790,3.57,3.0,14


,data,tipo_refeicao,curso,aberto,extracao_completa,quantidade_servida,valor_observado
0,2019-01-01,almoco,CURSO_TECNICO_A,True,True,0,NaN
1,2019-01-01,almoco,CURSO_TECNICO_B,True,True,6,NaN
2,2019-01-01,almoco,CURSO_TECNICO_C,True,True,1,NaN
3,2019-01-01,almoco,CURSO_TECNICO_D,True,True,3,NaN
4,2019-01-01,almoco,CURSO_TECNICO_E,True,True,0,NaN


## 4. Separar passado e futuro; preparar as entradas

O uso pretendido é prever **amanhã**, repetidamente. Por isso, o treino termina em 2024, a validação usa 2025 e o teste de 2026 fica reservado. A Entrega 3 citava 2025 como histórico e 2026 para teste; o mock inclui anos anteriores e permite esta validação adicional. Com dados reais, altere os cortes na primeira célula de código para refletir a data de implantação.

Só entram no conjunto modelável dias **abertos e com extração completa**. A variável aberto é usada apenas como filtro de cobertura, não como preditora.

### Variáveis conhecidas na véspera

O modelo usa o dia da semana, o mês, o tipo de refeição e o curso ativo na data prevista, desde que essas informações estejam disponíveis na véspera. A coluna data é usada para derivar variáveis de calendário, não entra diretamente no estimador. Quantidade servida é o alvo e não pode ser uma entrada.

- **Numéricas:** dia_semana e mes, derivadas do calendário; passam por imputação pela mediana e padronização.
- **Categóricas:** tipo_refeicao e curso; passam por imputação pela moda e codificação one-hot.
- **Não usadas como preditoras:** cadastro_cursos serve à auditoria de consistência entre tabelas. É montado com a tabela completa de estudantes, mas não alimenta o modelo. A variável curso usada como preditora tem categorias aprendidas pelo OneHotEncoder somente no treino.

No mock, aberto vem do calendário de cobertura e indica segunda a sábado. Não é calculado a partir dos eventos da própria data prevista. Para dados reais, o calendário de abertura precisa ser fornecido e confirmado pelo RU; recesso acadêmico não significa automaticamente fechamento do restaurante.

### Pipeline e ajuste

O ColumnTransformer e o RandomForestRegressor estão encapsulados em um único Pipeline. O ajuste (fit) ocorre somente no treino; a validação recebe previsões e o teste recebe apenas transformações, permanecendo reservado para a avaliação final. O Random Forest foi configurado com 300 árvores e semente 2026, como proposta inicial a confirmar com a equipe. A avaliação reporta MAE e RMSE na validação de 2025, na mesma partição dos baselines. A validação cruzada temporal usa apenas o treino até 2024, com todas as linhas de cada data juntas. O teste de 2026 não participa da escolha de parâmetros.

**O que pode falhar:** mediana e moda supõem que poucos valores estejam ausentes; muitas ausências indicam problema na fonte. A padronização das variáveis numéricas é mantida no pré-processador compartilhado, embora o Random Forest não dependa da escala das entradas. Categoria nova é aceita pelo codificador, mas pode indicar curso ou refeição sem histórico. Erros concentrados em meses ou cursos são sinais para rever essas escolhas.


In [4]:
modelavel = base_curso.loc[base_curso.aberto & base_curso.extracao_completa].copy()
treino = modelavel.loc[modelavel.data.le(FIM_TREINO)].copy()
validacao = modelavel.loc[modelavel.data.gt(FIM_TREINO) & modelavel.data.le(FIM_VALIDACAO)].copy()
teste = modelavel.loc[modelavel.data.gt(FIM_VALIDACAO)].copy()
if any(t.empty for t in [treino, validacao, teste]):
    raise ValueError("Treino, validação e teste precisam ter linhas. Revise os cortes de data.")
assert treino.data.max() < validacao.data.min() < teste.data.min()

def entradas(tabela):
    return pd.DataFrame({
        "dia_semana": tabela.data.dt.day_of_week.to_numpy(),
        "mes": tabela.data.dt.month.to_numpy(),
        "tipo_refeicao": tabela.tipo_refeicao.to_numpy(),
        "curso": tabela.curso.to_numpy(),
    }, index=tabela.index)

X_treino, X_validacao, X_teste = (entradas(t) for t in [treino, validacao, teste])
y_treino = treino["quantidade_servida"].astype("int64")
if y_treino.isna().any():
    raise ValueError("O alvo de treino tem valores ausentes.")

preprocessador = ColumnTransformer([
    ("numericas", Pipeline([
        ("faltantes", SimpleImputer(strategy="median")),
        ("escala", StandardScaler()),
    ]), ["dia_semana", "mes"]),
    ("categoricas", Pipeline([
        ("faltantes", SimpleImputer(strategy="most_frequent")),
        ("codigos", OneHotEncoder(handle_unknown="ignore")),
    ]), ["tipo_refeicao", "curso"]),
])

SEMENTE_MODELO = 2026
pipeline_modelo = Pipeline([
    ("preprocessador", preprocessador),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=SEMENTE_MODELO,
        n_jobs=-1,
    )),
])

# O fit do Pipeline ajusta imputação, escala, categorias e estimador somente no treino.
pipeline_modelo.fit(X_treino, y_treino)

# Validação pode receber previsões; o teste permanece reservado e recebe apenas transform.
preprocessador_ajustado = pipeline_modelo.named_steps["preprocessador"]
X_treino_pronto = preprocessador_ajustado.transform(X_treino)
X_validacao_pronta = preprocessador_ajustado.transform(X_validacao)
X_teste_pronto = preprocessador_ajustado.transform(X_teste)
previsao_modelo_validacao = pipeline_modelo.predict(X_validacao)

assert X_treino_pronto.shape[1] == X_validacao_pronta.shape[1] == X_teste_pronto.shape[1]

imputador_numerico = (
    preprocessador_ajustado.named_transformers_["numericas"]
    .named_steps["faltantes"]
)
escala_numerica = (
    preprocessador_ajustado.named_transformers_["numericas"]
    .named_steps["escala"]
)
codificador = (
    preprocessador_ajustado.named_transformers_["categoricas"]
    .named_steps["codigos"]
)

print("Linhas treino / validação / teste:", len(treino), len(validacao), len(teste))
print("Atributos após o pré-processamento:", X_treino_pronto.shape[1])
print("Medianas aprendidas no treino:", imputador_numerico.statistics_)
print("Escalas aprendidas no treino:", escala_numerica.scale_)
print("Categorias aprendidas no treino:", {
    nome: categorias.tolist()
    for nome, categorias in zip(["tipo_refeicao", "curso"], codificador.categories_)
})
print("Árvores do Random Forest:", pipeline_modelo.named_steps["regressor"].n_estimators)
print("Semente do Random Forest:", pipeline_modelo.named_steps["regressor"].random_state)
print("Previsões geradas para validação:", len(previsao_modelo_validacao))
print("A avaliação do modelo está na célula seguinte: validação cruzada temporal e métricas/resíduos de 2025.")

# Uma categoria inédita é ignorada sem alterar a dimensão das entradas.
novo_curso = X_validacao.iloc[[0]].copy()
novo_curso["curso"] = "CURSO_NOVO"
assert preprocessador_ajustado.transform(novo_curso).shape[1] == X_treino_pronto.shape[1]


Linhas treino / validação / teste: 33822 5634 4914
Atributos após o pré-processamento: 11
Medianas aprendidas no treino: [2. 7.]
Escalas aprendidas no treino: [1.70772106 3.447839  ]
Categorias aprendidas no treino: {'tipo_refeicao': ['almoco', 'cafe_da_manha', 'jantar'], 'curso': ['CURSO_TECNICO_A', 'CURSO_TECNICO_B', 'CURSO_TECNICO_C', 'CURSO_TECNICO_D', 'CURSO_TECNICO_E', 'CURSO_TECNICO_F']}
Árvores do Random Forest: 300
Semente do Random Forest: 2026
Previsões geradas para validação: 5634
Métricas do modelo serão calculadas na comparação da equipe, na mesma validação dos baselines.


### Avaliação do Random Forest

A validação cruzada temporal usa apenas o conjunto de treino (2019–2024). Cada fold separa datas inteiras: todas as linhas de uma mesma data permanecem juntas, mesmo havendo várias combinações de curso e refeição. O Pipeline é reajustado dentro de cada fold, sem usar validação ou teste.

Depois da validação cruzada, o Pipeline final já ajustado no treino gera previsões para a validação de 2025. Calculamos MAE e RMSE e analisamos os resíduos por curso e tipo de refeição. Resíduo positivo significa que o modelo previu menos refeições que o valor observado; negativo significa que previu mais.

O teste de 2026 continua reservado. Ele não é usado para escolher parâmetros. Nesta versão, a configuração do Random Forest (300 árvores) é fixa; a validação cruzada serve para avaliar a estabilidade temporal no período de treino. Se a equipe decidir ajustar hiperparâmetros, essa escolha deve usar somente os folds temporais do treino.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Separa as datas do treino em blocos temporais, mantendo todas as linhas
# de cada dia juntas (cursos e refeições não são separados entre folds).
datas_treino = np.sort(treino["data"].dropna().unique())
separador_temporal = TimeSeriesSplit(n_splits=5)
folds_temporais = []

for indices_treino, indices_validacao in separador_temporal.split(datas_treino):
    datas_fold_treino = datas_treino[indices_treino]
    datas_fold_validacao = datas_treino[indices_validacao]

    linhas_fold_treino = np.flatnonzero(
        treino["data"].isin(datas_fold_treino).to_numpy()
    )
    linhas_fold_validacao = np.flatnonzero(
        treino["data"].isin(datas_fold_validacao).to_numpy()
    )

    # Verificações: nenhuma data aparece nos dois lados e a ordem é temporal.
    assert set(datas_fold_treino).isdisjoint(set(datas_fold_validacao))
    assert datas_fold_treino.max() < datas_fold_validacao.min()
    folds_temporais.append((linhas_fold_treino, linhas_fold_validacao))

# cross_validate clona o Pipeline para cada fold e ajusta o pré-processador
# e o Random Forest só com as linhas de treino daquele fold.
resultado_cv = cross_validate(
    pipeline_modelo,
    X_treino,
    y_treino,
    cv=folds_temporais,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
    },
    n_jobs=1,
)

resumo_cv = pd.DataFrame({
    "fold": np.arange(1, len(folds_temporais) + 1),
    "MAE": -resultado_cv["test_MAE"],
    "RMSE": -resultado_cv["test_RMSE"],
})
print("=== VALIDAÇÃO CRUZADA TEMPORAL — SOMENTE TREINO ===")
print("Intervalo de datas do treino:", datas_treino.min(), "a", datas_treino.max())
display(resumo_cv.round(2))
print("Média dos folds:")
display(resumo_cv[["MAE", "RMSE"]].mean().to_frame("média").round(2))

# Avalia o modelo final no período de validação (2025).
# Essas linhas não participaram do fit do Pipeline nem da validação cruzada.
y_validacao = validacao["quantidade_servida"].astype("int64")
previsao_validacao = np.asarray(previsao_modelo_validacao)

metricas_modelo_validacao = pd.DataFrame([{
    "conjunto": "validação 2025",
    "linhas": len(validacao),
    "MAE": mean_absolute_error(y_validacao, previsao_validacao),
    "RMSE": np.sqrt(mean_squared_error(y_validacao, previsao_validacao)),
}]).round(2)

print("=== RANDOM FOREST — VALIDAÇÃO 2025 ===")
display(metricas_modelo_validacao)

# Resíduo = real - previsão. O erro absoluto facilita comparar os grupos.
diagnostico_modelo = validacao[
    ["data", "curso", "tipo_refeicao", "quantidade_servida"]
].copy()
diagnostico_modelo["previsao"] = previsao_validacao
diagnostico_modelo["residuo"] = (
    diagnostico_modelo["quantidade_servida"] - diagnostico_modelo["previsao"]
)
diagnostico_modelo["erro_absoluto"] = diagnostico_modelo["residuo"].abs()

def resumo_erros_por(grupos):
    return (
        diagnostico_modelo
        .groupby(grupos)
        .agg(
            linhas=("residuo", "size"),
            MAE=("erro_absoluto", "mean"),
            RMSE=("residuo", lambda valores: np.sqrt(np.mean(valores ** 2))),
            vies_medio=("residuo", "mean"),
        )
        .round(2)
        .sort_values("MAE", ascending=False)
    )

print("=== ERROS POR CURSO E TIPO DE REFEIÇÃO ===")
display(resumo_erros_por(["curso", "tipo_refeicao"]))
print("=== ERROS POR CURSO ===")
display(resumo_erros_por(["curso"]))
print("=== ERROS POR TIPO DE REFEIÇÃO ===")
display(resumo_erros_por(["tipo_refeicao"]))

# Registro explícito dos conjuntos usados. O teste de 2026 não é avaliado aqui.
print("=== DADOS USADOS ===")
print(f"Ajuste final do Pipeline: treino até {treino.data.max().date()} ({len(treino):,} linhas).")
print(f"Validação e métricas: 2025 ({len(validacao):,} linhas).")
print(f"Teste: {len(teste):,} linhas; reservado, sem métricas nem escolha de parâmetros nesta etapa.")

## 5. Verificar vazamento

| Variável | Uso na previsão | Motivo |
|---|---|---|
| Data, dia da semana e mês | Sim | O calendário é conhecido na véspera; a data bruta não entra diretamente no modelo. |
| aberto | Somente como filtro de cobertura | No mock vem do calendário de funcionamento, não dos eventos do dia previsto. |
| cadastro_cursos | Somente auditoria | É construído com a tabela completa para conferir consistência e não entra em X. |
| tipo_refeicao | Sim | É conhecida para cada previsão por data e refeição. |
| curso | Sim, se ativo e conhecido na véspera | Entra como categoria; o OneHotEncoder aprende categorias somente no treino. |
| quantidade_servida em D | Não | É o próprio alvo, conhecido depois. |
| Valor observado em D | Não | Também depende do consumo futuro. |
| Faltas e notas finais do semestre | Não | Podem incorporar informação posterior à véspera. |
| Refeições servidas em D-7 | Sim, se a catraca estiver atualizada | Já ocorreram antes de D-1; o protocolo deve ser previsão diária atualizada. |
| Refeições preparadas e sobras em D | Não | Resultam da decisão de produção. |
| Estatísticas ajustadas com validação ou teste | Não | O pré-processador aprende apenas no treino. |

Os diagnósticos individuais de três variáveis por integrante estão no relatório de qualidade e precisam da confirmação pessoal de cada um. Os baselines são avaliados independentemente do pré-processador. O modelo de regressão usa o Pipeline, ajustado no treino e aplicado à validação; o teste continua reservado.


## 6. Comparar dois baselines, sem tocar no teste

**Trivial:** média do treino para cada curso e refeição. **D-7:** repete a quantidade do mesmo curso e refeição exatamente sete dias antes. D-7 é uma **regra proposta para o piloto**; não sabemos qual regra o RU usa hoje.

O erro principal é o **MAE em refeições por curso e tipo**, fácil de comparar com a quantidade que se planeja preparar. O RMSE mostra se existem erros grandes. Sem capacidade `k` e custos `B/C` confirmados, não dá para chamar MAE de custo econômico nem escolher uma regra ótima para subsídios. D-7 só pode usar dias de 2025 anteriores à data prevista se a avaliação representar previsões feitas **dia após dia**, com o histórico atualizado. Se faltar histórico ou curso novo, mostramos quantas linhas não puderam ser comparadas.

In [5]:
# A média aprende apenas com 2019–2024.
media = (treino.groupby(["curso", "tipo_refeicao"]).quantidade_servida.mean()
         .reset_index(name="previsao_media"))

# O deslocamento é por data, não por posição: uma data ausente não vira D-7 por engano.
anterior = base_curso[["data", "tipo_refeicao", "curso", "quantidade_servida"]].copy()
anterior["data"] = anterior.data + pd.Timedelta(days=7)
anterior = anterior.rename(columns={"quantidade_servida": "previsao_d7"})
avaliacao = validacao.merge(media, how="left", validate="many_to_one",
                            on=["curso", "tipo_refeicao"])
avaliacao = avaliacao.merge(anterior, how="left", validate="one_to_one",
                            on=["data", "tipo_refeicao", "curso"])
comparaveis = avaliacao.dropna(subset=["previsao_media", "previsao_d7"]).copy()
if comparaveis.empty:
    raise ValueError("Nenhuma linha da validação tem os dois baselines. Revise cobertura e datas.")

def erros(tabela, coluna):
    diferenca = tabela.quantidade_servida - tabela[coluna]
    return {"linhas": len(tabela), "MAE": round(diferenca.abs().mean(), 2),
            "RMSE": round(np.sqrt((diferenca ** 2).mean()), 2)}

resultado = []
for nome, coluna in [("Média do treino", "previsao_media"), ("D-7 proposto", "previsao_d7")]:
    for (curso, refeicao), grupo in comparaveis.groupby(["curso", "tipo_refeicao"]):
        resultado.append({"baseline": nome, "curso": curso, "refeicao": refeicao,
                          **erros(grupo, coluna)})
    resultado.append({"baseline": nome, "curso": "TODOS", "refeicao": "TODAS",
                      **erros(comparaveis, coluna)})
metricas = pd.DataFrame(resultado)

# Total diário só usa datas com todas as combinações elegíveis para a comparação.
linhas_por_data = validacao.groupby("data").size()
linhas_comparaveis = comparaveis.groupby("data").size()
datas_completas = linhas_por_data.index[linhas_por_data.eq(linhas_comparaveis.reindex(linhas_por_data.index, fill_value=0))]
totais = (comparaveis.loc[comparaveis.data.isin(datas_completas)]
          .groupby("data")[["quantidade_servida", "previsao_media", "previsao_d7"]]
          .sum().reset_index())
metricas_total = pd.DataFrame([
    {"baseline": nome, **erros(totais, coluna)}
    for nome, coluna in [("Média do treino", "previsao_media"), ("D-7 proposto", "previsao_d7")]
]) if len(totais) else pd.DataFrame()

metricas.to_csv(PROCESSADOS / "resultados_baselines_validacao_por_curso.csv", index=False)
metricas_total.to_csv(PROCESSADOS / "resultados_baselines_total_diario.csv", index=False)
print(f"Validação: {len(validacao)} linhas; {len(comparaveis)} com os dois baselines; "
      f"{len(totais)} dias com total comparável.")
print("Erro geral por linha de curso/refeição:")
display(metricas.loc[metricas.curso.eq("TODOS")])
print("Erro por curso e refeição:")
display(metricas.loc[metricas.curso.ne("TODOS")])
print("Erro do total diário:")
display(metricas_total)

Validação: 5634 linhas; 5634 com os dois baselines; 313 dias com total comparável.
Erro geral por linha de curso/refeição:


,baseline,curso,refeicao,linhas,MAE,RMSE
18,Média do treino,TODOS,TODAS,5634,2.67,3.24
37,D-7 proposto,TODOS,TODAS,5634,2.14,2.95


Erro por curso e refeição:


,baseline,curso,refeicao,linhas,MAE,RMSE
0,Média do treino,CURSO_TECNICO_A,almoco,313,3.96,4.42
1,Média do treino,CURSO_TECNICO_A,cafe_da_manha,313,2.09,2.45
2,Média do treino,CURSO_TECNICO_A,jantar,313,1.77,2.07
3,Média do treino,CURSO_TECNICO_B,almoco,313,4.11,4.55
4,Média do treino,CURSO_TECNICO_B,cafe_da_manha,313,2.14,2.61
5,Média do treino,CURSO_TECNICO_B,jantar,313,2.10,2.48
6,Média do treino,CURSO_TECNICO_C,almoco,313,3.96,4.42
7,Média do treino,CURSO_TECNICO_C,cafe_da_manha,313,2.16,2.59
8,Média do treino,CURSO_TECNICO_C,jantar,313,1.80,2.15
9,Média do treino,CURSO_TECNICO_D,almoco,313,4.01,4.51


Erro do total diário:


,baseline,linhas,MAE,RMSE
0,Média do treino,313,42.91,45.10
1,D-7 proposto,313,15.04,27.19


## 7. O que esta entrega prova — e o que ainda não prova

O notebook gera uma base por curso, com zero apenas quando a cobertura é conhecida, e avalia baselines e RandomForestRegressor na validação de 2025. A validação cruzada do modelo é temporal, usa somente o treino e mantém juntas as linhas de cada data. O Pipeline final é ajustado apenas no treino; a validação recebe previsões e o teste de 2026 permanece reservado, sem ser usado para escolher parâmetros. O total do RU é a soma dos cursos, não um segundo alvo treinado.

cadastro_cursos é usado apenas na auditoria de integridade. As preditoras do modelo são calendário, tipo de refeição e curso ativo. Notas e faltas consolidadas não entram, pois podem conter informação posterior à véspera.

O mock demonstra que o processo pode ser reproduzido; não demonstra desempenho no RU real. Antes de aplicar aos dados institucionais, é preciso confirmar o calendário, a cobertura das extrações, o significado dos valores monetários, a regra de planejamento atualmente usada, a capacidade, os custos e as permissões de acesso. Um curso novo pode ser transformado pelo pré-processador, mas precisa de histórico para o baseline por curso; o notebook deve mostrar a perda de cobertura em vez de inventar previsão.

**Uso de IA generativa:** Codex (OpenAI) auxiliou a coordenação na consolidação deste notebook, na redação explicativa e na revisão dos blocos de auditoria, cobertura, partição, pré-processamento, Pipeline com RandomForestRegressor e baselines. A equipe deve conferir os resultados e ser capaz de explicar cada trecho. As declarações de cada notebook individual permanecem responsabilidade de seu autor.

**Saídas reproduzíveis:** dados/processado/base_analitica_diaria_curso.csv, base_analitica_diaria.csv, resultados_baselines_validacao_por_curso.csv e resultados_baselines_total_diario.csv.
